# 🌿 HerbaQuest — Reconnaissance de plantes pour la réalité augmentée éducative

**Projet 4 — Module Deep Learning 2026/2027 — Master IADAIG, ENS de Fès (USMBA)**
Ibtissam Draoui · Aya Laazizi · Majda Bakhari

**Question de recherche.** Pour une application d'AR éducative sur mobile, quel modèle offre le meilleur
compromis **précision / robustesse / latence** ? Et comment transformer l'**incertitude** du modèle en
mécanique de jeu pédagogique (carte débloquée, mini-quiz, « rescanne ») ?

| Section | Contenu | Barème visé |
|---|---|---|
| 1 | Configuration et reproductibilité | Code (15 %) |
| 2 | Exploration des données (EDA) | EDA (20 %) |
| 3 | Nettoyage, découpage, augmentation | Prétraitement (20 %) |
| 4 | 4 architectures, ablation, Optuna, fine-tuning | Modèles (20 %) |
| 5 | Métriques, tests statistiques, calibration, Grad-CAM, erreurs, robustesse | Évaluation (25 %) |
| 6 | Export TFLite et fichiers pour la démo et le papier | Code (15 %) |

**Avant d'exécuter (Kaggle)** : *Add Input* → dataset **House Plant Species** ; *Accelerator* → **GPU T4** ;
*Internet* → **On** (poids ImageNet). Lancez d'abord avec `QUICK_RUN = True` (≈ 5 min) pour vérifier que tout
tourne, puis `QUICK_RUN = False` pour les vrais résultats.

## 1. Configuration et reproductibilité

In [ ]:
import os, io, gc, json, time, math, random, hashlib, shutil, warnings
from pathlib import Path
from itertools import islice
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageEnhance, ImageFilter, ImageOps
from scipy.special import softmax, log_softmax
from scipy.optimize import minimize_scalar
from scipy.stats import binomtest

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.manifold import TSNE
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             confusion_matrix, classification_report,
                             roc_curve, auc, roc_auc_score, top_k_accuracy_score)
from sklearn.preprocessing import label_binarize

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")   # masque les messages techniques du GPU (sans effet sur les calculs)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

try:
    from tqdm.auto import tqdm
except ImportError:                      # tqdm est facultatif
    tqdm = lambda x, **k: x

warnings.filterwarnings("ignore", category=UserWarning)
Image.MAX_IMAGE_PIXELS = None            # certaines photos web sont très grandes

# ----------------------------- Paramètres globaux -----------------------------
SEED         = 42
QUICK_RUN    = False     # True : sous-échantillon + peu d'époques, pour tester le notebook en ~5 min
IMG_SIZE     = 224       # taille d'entrée standard des modèles ImageNet
CACHE_SIZE   = 256       # taille du cache disque (on recadre ensuite en 224)
BATCH        = 32
EPOCHS_HEAD  = 8         # phase 1 : tête de classification seule (base gelée)
EPOCHS_FT    = 12        # phase 2 : fine-tuning des couches hautes
EPOCHS_CNN   = 25        # CNN de référence entraîné de zéro
N_TRIALS     = 12        # essais Optuna
EPOCHS_TRIAL = 4
FT_FRACTION  = 0.30      # part des couches de la base dégelées en phase 2
LR_FT        = 1e-5      # taux d'apprentissage du fine-tuning (faible : on ajuste, on ne ré-apprend pas)
N_ROBUST     = 600       # images de test utilisées pour les tests de robustesse
MODEL_NAMES  = ["cnn_scratch", "mobilenetv2", "resnet50v2", "efficientnetb0"]

if QUICK_RUN:
    EPOCHS_HEAD, EPOCHS_FT, EPOCHS_CNN, N_TRIALS, EPOCHS_TRIAL, N_ROBUST = 2, 2, 3, 2, 1, 120

ON_KAGGLE = Path("/kaggle/input").exists()
WORK      = Path("/kaggle/working") if ON_KAGGLE else Path("./herbaquest_out")
FIG_DIR, CACHE_DIR, MODEL_DIR, EXPORT_DIR, PAPER_DIR = [WORK / d for d in
    ("figures", "cache", "models", "export", "paper_assets")]
for d in (FIG_DIR, CACHE_DIR, MODEL_DIR, EXPORT_DIR, PAPER_DIR):
    d.mkdir(parents=True, exist_ok=True)

def set_seed(seed=SEED):
    """Fixe toutes les graines. Sur GPU, quelques opérations restent non déterministes :
    les résultats peuvent varier de quelques dixièmes de point d'une exécution à l'autre."""
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed()
print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
print("GPU :", tf.config.list_physical_devices("GPU") or "aucun (l'entraînement sera très lent)")

### Charte graphique
Une seule charte pour toutes les figures (notebook, papier, diapositives) : une couleur **fixe** par modèle,
un dégradé bleu unique pour les grandeurs, des axes discrets. Chaque figure est enregistrée en PNG (300 dpi) et en PDF.

In [ ]:
INK, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
BLUE, ORANGE, AQUA, YELLOW, MAGENTA, GREEN, VIOLET, RED = (
    "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948")
MODEL_LABEL = {"cnn_scratch": "CNN de référence", "mobilenetv2": "MobileNetV2",
               "resnet50v2": "ResNet50V2", "efficientnetb0": "EfficientNetB0"}
MODEL_COLOR = {"cnn_scratch": YELLOW, "mobilenetv2": BLUE, "resnet50v2": ORANGE, "efficientnetb0": AQUA}
SEQ_CMAP = "Blues"

mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "font.family": "DejaVu Sans", "font.size": 10, "text.color": INK,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.labelcolor": MUTED, "axes.edgecolor": GRID, "axes.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelsize": 9, "ytick.labelsize": 9,
    "lines.linewidth": 2, "lines.markersize": 6, "legend.frameon": False, "figure.dpi": 110,
})

def savefig(name):
    """Enregistre la figure courante dans figures/ (PNG pour les slides, PDF pour LaTeX)."""
    plt.savefig(FIG_DIR / f"{name}.png", dpi=300, bbox_inches="tight")
    plt.savefig(FIG_DIR / f"{name}.pdf", bbox_inches="tight")
    plt.show()

def per_class(frame, k, col="label", seed=SEED):
    """Au plus k lignes tirées au hasard par classe (reproductible)."""
    return frame.sample(frac=1, random_state=seed).groupby(col, sort=False).head(k).sort_values(col).reset_index(drop=True)

def short(label, n=22):
    """Nom de classe raccourci pour les axes (« Aloe Vera (…) » → « Aloe Vera »)."""
    s = label.split("(")[0].strip()
    return s if len(s) <= n else s[: n - 1] + "…"

## 2. Exploration des données (EDA)
### 2.1 Localisation et indexation
Le dossier du dataset est détecté automatiquement : on cherche le répertoire qui contient le plus de
sous-dossiers d'images (un sous-dossier = une espèce). Hors Kaggle, définissez la variable d'environnement
`HERBAQUEST_DATA` vers le dossier décompressé.

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif", ".jfif"}

def has_images(folder, probe=30):
    return any(f.suffix.lower() in IMG_EXT for f in islice(folder.iterdir(), probe) if f.is_file())

def find_dataset_root(bases):
    """Renvoie le dossier contenant le plus de sous-dossiers d'images."""
    best, best_n = None, 0
    for base in map(Path, bases):
        if not base.exists():
            continue
        for d in [base, *[p for p in base.rglob("*") if p.is_dir()]]:
            try:
                n = sum(1 for s in d.iterdir() if s.is_dir() and has_images(s))
            except PermissionError:
                continue
            if n > best_n:
                best, best_n = d, n
    if best is None or best_n < 2:
        raise FileNotFoundError("Dataset introuvable : ajoutez « House Plant Species » au notebook "
                                "(Add Input) ou définissez HERBAQUEST_DATA.")
    return best

DATA_ROOT = find_dataset_root([os.environ.get("HERBAQUEST_DATA", "/kaggle/input")])
rows = [(str(f), d.name) for d in sorted(DATA_ROOT.iterdir()) if d.is_dir()
        for f in sorted(d.iterdir()) if f.suffix.lower() in IMG_EXT]
index_df = pd.DataFrame(rows, columns=["path", "label"])

if QUICK_RUN:   # 8 classes x 60 images au plus
    keep = sorted(index_df["label"].unique())[:8]
    index_df = per_class(index_df[index_df["label"].isin(keep)], 60)

print(f"Racine : {DATA_ROOT}\n{len(index_df):,} fichiers image · {index_df['label'].nunique()} classes")

### 2.2 Audit image par image (et mise en cache)
Une seule passe sur le disque fait deux choses :
1. **mesurer** chaque image (dimensions, format, poids, luminosité, contraste, netteté, couleur, empreintes) ;
2. **mettre en cache** une version 256×256 (orientation EXIF corrigée, petit côté ramené à 256 px, recadrage central).

*Pourquoi un cache ?* Le dataset pèse plusieurs Go : décoder les originaux à chaque époque dominerait le temps
d'entraînement. *Pourquoi 256 puis 224 ?* C'est le protocole ImageNet classique : le recadrage aléatoire 224
dans 256 sert d'augmentation, le recadrage central sert à l'évaluation.
Les mesures de qualité sont calculées sur la version 256 px : elles sont donc comparables d'une image à l'autre.

In [ ]:
def laplacian_var(gray):
    """Netteté : variance du laplacien (faible = image floue)."""
    lap = (-4 * gray[1:-1, 1:-1] + gray[:-2, 1:-1] + gray[2:, 1:-1] + gray[1:-1, :-2] + gray[1:-1, 2:])
    return float(lap.var())

def dhash(im, size=16):
    """Empreinte perceptuelle (difference hash, 256 bits) : identique pour deux copies
    redimensionnées ou recompressées de la même photo."""
    g = np.asarray(im.convert("L").resize((size + 1, size), Image.BILINEAR), dtype=np.int16)
    bits = (g[:, 1:] > g[:, :-1]).flatten()
    return np.packbits(bits).tobytes().hex()

def process_one(args):
    idx, path, label = args
    rec = {"idx": idx, "path": path, "label": label, "ok": False}
    try:
        raw = Path(path).read_bytes()
        rec["bytes"], rec["md5"] = len(raw), hashlib.md5(raw).hexdigest()
        im = Image.open(io.BytesIO(raw))
        rec["format"], rec["mode"] = im.format, im.mode
        rec["width"], rec["height"] = im.size
        if im.format == "JPEG":                     # décodage JPEG à échelle réduite : beaucoup plus rapide
            im.draft("RGB", (CACHE_SIZE * 2, CACHE_SIZE * 2))
        im = ImageOps.exif_transpose(im).convert("RGB")
        w, h = im.size
        s = CACHE_SIZE / min(w, h)
        im = im.resize((max(CACHE_SIZE, round(w * s)), max(CACHE_SIZE, round(h * s))), Image.BILINEAR)
        w, h = im.size
        l, t = (w - CACHE_SIZE) // 2, (h - CACHE_SIZE) // 2
        im = im.crop((l, t, l + CACHE_SIZE, t + CACHE_SIZE))
        out = CACHE_DIR / f"{idx:06d}.jpg"
        im.save(out, "JPEG", quality=92)
        a = np.asarray(im, dtype=np.float32)
        gray = a @ np.array([0.299, 0.587, 0.114], dtype=np.float32)
        mx, mn = a.max(axis=2), a.min(axis=2)
        rec.update(cache_path=str(out), brightness=float(gray.mean()), contrast=float(gray.std()),
                   sharpness=laplacian_var(gray), saturation=float(((mx - mn) / (mx + 1e-6)).mean()),
                   mean_r=float(a[..., 0].mean()), mean_g=float(a[..., 1].mean()),
                   mean_b=float(a[..., 2].mean()), dhash=dhash(im), ok=True)
    except Exception as e:                          # image illisible : on la note, on continue
        rec["error"] = f"{type(e).__name__}: {e}"[:120]
    return rec

META_CSV = WORK / "eda_metadata.csv"
meta = pd.read_csv(META_CSV, dtype={"dhash": str, "md5": str}) if META_CSV.exists() else None
if meta is not None and len(meta) == len(index_df) and meta["cache_path"].dropna().map(os.path.exists).all():
    print("Métadonnées rechargées depuis", META_CSV)
else:
    jobs = [(i, p, l) for i, (p, l) in enumerate(zip(index_df["path"], index_df["label"]))]
    with ThreadPoolExecutor(max_workers=os.cpu_count() or 4) as ex:
        meta = pd.DataFrame(list(tqdm(ex.map(process_one, jobs), total=len(jobs), desc="Audit + cache")))
    meta.to_csv(META_CSV, index=False)

meta["megapixels"] = meta["width"] * meta["height"] / 1e6
meta["aspect"] = meta["width"] / meta["height"]
n_bad = int((~meta["ok"]).sum())
print(f"{len(meta):,} fichiers audités · {n_bad} illisibles")
display(meta[meta["ok"]][["width", "height", "megapixels", "bytes", "brightness", "contrast",
                           "sharpness", "saturation"]].describe().T.round(2))
print("\nFormats :", meta["format"].value_counts(dropna=False).to_dict())
print("Modes   :", meta["mode"].value_counts(dropna=False).to_dict())

### 2.3 Distribution des classes et déséquilibre
Le rapport max/min mesure le déséquilibre. S'il est élevé, l'accuracy seule est trompeuse : on suivra aussi le
**F1 macro** et l'**accuracy équilibrée**, et on pondérera la fonction de perte (section 3).

In [ ]:
ok = meta[meta["ok"]].copy()
counts = ok["label"].value_counts()
IMBALANCE_RATIO = counts.max() / counts.min()

fig, ax = plt.subplots(figsize=(8, max(4, 0.2 * len(counts))))
ax.barh([short(c) for c in counts.index[::-1]], counts.values[::-1], color=BLUE, height=0.72)
ax.axvline(counts.median(), color=MUTED, lw=1, ls="--")
ax.text(counts.median(), len(counts) - 0.3, f" médiane = {int(counts.median())}", color=MUTED, fontsize=9, va="bottom")
ax.set_xlabel("Nombre d'images"); ax.grid(axis="y", visible=False)
ax.set_title(f"Distribution des {len(counts)} classes — rapport max/min = {IMBALANCE_RATIO:.1f}")
ax.tick_params(axis="y", labelsize=7.5)
savefig("fig_class_distribution")
print(f"Classe la plus fournie : {counts.index[0]} ({counts.iloc[0]}) · la plus rare : {counts.index[-1]} ({counts.iloc[-1]})")

### 2.4 Aperçu visuel
Une image par classe (tirage aléatoire reproductible). À regarder : variété des cadrages (plante entière, gros
plan), arrière-plans, présence de texte ou de montages, classes visuellement proches.

In [ ]:
def show_grid(frame, title, ncols=8, titles=None, name=None, size=1.45):
    n = len(frame); nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * size, nrows * size * 1.14))
    for ax in np.atleast_1d(axes).ravel():
        ax.axis("off")
    for k, (ax, (_, r)) in enumerate(zip(np.atleast_1d(axes).ravel(), frame.iterrows())):
        ax.imshow(Image.open(r["cache_path"]))
        ax.set_title(titles[k] if titles else short(r["label"], 16), fontsize=6.5, loc="center", fontweight="normal")
    fig.suptitle(title, x=0.01, ha="left", fontweight="bold", fontsize=12)
    plt.tight_layout()
    savefig(name) if name else plt.show()

sample = per_class(ok, 1)
show_grid(sample, "Une image par classe", name="fig_samples")

### 2.5 Dimensions, formats et poids
Des images très petites portent peu d'information une fois agrandies à 224 px ; des rapports largeur/hauteur
extrêmes perdent du contenu au recadrage central. Ces deux constats motivent le filtrage de la section 3.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
axes[0].scatter(ok["width"], ok["height"], s=5, alpha=0.25, color=BLUE, linewidths=0)
axes[0].set_xscale("log"); axes[0].set_yscale("log")
axes[0].set_xlabel("Largeur (px)"); axes[0].set_ylabel("Hauteur (px)"); axes[0].set_title("Résolutions d'origine")
axes[1].hist(np.clip(ok["aspect"], 0.3, 3), bins=50, color=BLUE)
axes[1].axvline(1, color=MUTED, lw=1, ls="--"); axes[1].set_xlabel("Rapport largeur / hauteur"); axes[1].set_title("Formats d'image")
axes[2].hist(np.log10(ok["megapixels"].clip(lower=1e-3)), bins=50, color=BLUE)
axes[2].set_xlabel("log10(mégapixels)"); axes[2].set_title("Taille des images")
plt.tight_layout(); savefig("fig_resolutions")
N_SMALL = int((ok[["width", "height"]].min(axis=1) < 96).sum())
N_EXTREME = int(((ok["aspect"] > 2.5) | (ok["aspect"] < 0.4)).sum())
print(f"Images dont le petit côté < 96 px : {N_SMALL} · rapports extrêmes (>2,5 ou <0,4) : {N_EXTREME}")

### 2.6 Qualité photométrique : luminosité, contraste, netteté
Ces mesures servent deux fois : ici pour connaître le dataset, et en section 5 pour tester si les **erreurs du
modèle se concentrent sur les images sombres ou floues** — ce qui justifiera (ou non) les tests de robustesse.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for ax, col, lab in zip(axes, ["brightness", "contrast", "sharpness"],
                        ["Luminosité moyenne (0–255)", "Contraste (écart-type)", "Netteté (log10 var. laplacien)"]):
    v = np.log10(ok[col] + 1) if col == "sharpness" else ok[col]
    ax.hist(v, bins=50, color=BLUE)
    for q in (0.05, 0.95):
        ax.axvline(v.quantile(q), color=MUTED, lw=1, ls="--")
    ax.set_xlabel(lab)
axes[0].set_title("Qualité photométrique (pointillés : 5ᵉ et 95ᵉ centiles)")
plt.tight_layout(); savefig("fig_quality")

show_grid(ok.nsmallest(8, "brightness"), "Les 8 images les plus sombres")
show_grid(ok.nsmallest(8, "sharpness"), "Les 8 images les plus floues")

In [ ]:
# Signature couleur par classe : des classes de même teinte dominante risquent d'être confondues.
color_sig = ok.groupby("label")[["mean_r", "mean_g", "mean_b", "saturation", "brightness"]].mean()
greenness = (color_sig["mean_g"] - (color_sig["mean_r"] + color_sig["mean_b"]) / 2).sort_values()
fig, ax = plt.subplots(figsize=(8, max(4, 0.2 * len(greenness))))
ax.barh([short(c) for c in greenness.index], greenness.values, height=0.72,
        color=[np.clip(color_sig.loc[c, ["mean_r", "mean_g", "mean_b"]].values / 255, 0, 1) for c in greenness.index],
        edgecolor=GRID, linewidth=0.5)
ax.set_xlabel("Dominante verte : V − (R + B)/2"); ax.grid(axis="y", visible=False); ax.tick_params(axis="y", labelsize=7.5)
ax.set_title("Signature couleur par classe (barre = couleur moyenne de la classe)")
savefig("fig_color_signature")

### 2.7 Doublons et conflits d'étiquettes
Images collectées sur le Web : la même photo peut apparaître plusieurs fois, parfois dans **deux classes
différentes**. Deux risques : la **fuite de données** (un doublon en entraînement et en test gonfle
artificiellement le score) et le **bruit d'étiquette**. On détecte les copies exactes (MD5) et les copies
redimensionnées ou recompressées (empreinte perceptuelle dHash).

In [ ]:
dup_md5 = ok[ok.duplicated("md5", keep=False)]
grp = ok.groupby("dhash")
ok["dup_size"] = grp["dhash"].transform("size")
ok["dup_nlabels"] = grp["label"].transform("nunique")
ok["dup_first"] = ~ok.duplicated("dhash", keep="first")
N_DUP_EXACT = int(ok.duplicated("md5", keep="first").sum())
N_DUP_NEAR = int((~ok["dup_first"]).sum())
N_CONFLICT = int((ok["dup_nlabels"] > 1).sum())
print(f"Copies exactes (MD5) en trop        : {N_DUP_EXACT}")
print(f"Copies exactes ou quasi (dHash)     : {N_DUP_NEAR}")
print(f"Images dans un groupe multi-classes : {N_CONFLICT}")
ex = ok[ok["dup_size"] > 1].sort_values(["dup_nlabels", "dhash"], ascending=[False, True]).head(8)
if len(ex):
    show_grid(ex, "Exemples de doublons (deux à deux)", name="fig_duplicates")

In [ ]:
# Figure de synthèse de l'EDA, au format du papier (pleine largeur, faible hauteur)
fig, axes = plt.subplots(1, 4, figsize=(13, 2.6))
axes[0].bar(range(len(counts)), counts.values, color=BLUE, width=0.8)
axes[0].axhline(counts.median(), color=MUTED, lw=1, ls="--")
axes[0].set_xlabel("Classes (triées)"); axes[0].set_ylabel("Images"); axes[0].set_xticks([])
axes[0].set_title(f"(a) Classes : max/min = {IMBALANCE_RATIO:.1f}")
axes[1].hist(np.log10(ok[["width", "height"]].min(axis=1)), bins=40, color=BLUE)
axes[1].set_xlabel("log10 du petit côté (px)"); axes[1].set_title("(b) Résolution")
axes[2].hist(ok["brightness"], bins=40, color=BLUE); axes[2].set_xlabel("Luminosité moyenne"); axes[2].set_title("(c) Luminosité")
axes[3].hist(np.log10(ok["sharpness"] + 1), bins=40, color=BLUE); axes[3].set_xlabel("log10 var. laplacien"); axes[3].set_title("(d) Netteté")
plt.tight_layout(); savefig("fig_eda_summary")

### 2.8 Séparabilité des classes avant tout entraînement
On projette en 2D (t-SNE) les descripteurs d'un MobileNetV2 **pré-entraîné sur ImageNet, non ré-entraîné**.
Un k-plus-proches-voisins sur ces descripteurs donne un **plancher** : si un modèle fine-tuné ne le dépasse pas
nettement, c'est que l'entraînement n'apporte rien. Les amas qui se chevauchent annoncent les confusions de la section 5.

In [ ]:
emb_sample = per_class(ok, 40)

def load_batch(paths, size=IMG_SIZE):
    """Charge des images du cache et les recadre au centre (numpy float32, 0–255)."""
    o = (CACHE_SIZE - size) // 2
    return np.stack([np.asarray(Image.open(p).convert("RGB"), dtype=np.float32)[o:o + size, o:o + size] for p in paths])

feat_net = keras.applications.MobileNetV2(include_top=False, weights="imagenet", pooling="avg",
                                          input_shape=(IMG_SIZE, IMG_SIZE, 3))
emb = np.concatenate([feat_net.predict(load_batch(emb_sample["cache_path"].values[i:i + 128]) / 127.5 - 1, verbose=0)
                      for i in range(0, len(emb_sample), 128)])
del feat_net; gc.collect()

KNN_BASELINE = cross_val_score(KNeighborsClassifier(5, metric="cosine"), emb, emb_sample["label"], cv=5).mean()
xy = TSNE(n_components=2, init="pca", perplexity=30, random_state=SEED).fit_transform(emb)

top6 = counts.index[:6].tolist()                 # les 6 classes les plus fournies
fig, ax = plt.subplots(figsize=(7.5, 6))
other = ~emb_sample["label"].isin(top6)
ax.scatter(xy[other, 0], xy[other, 1], s=10, color="#c9c8c2", linewidths=0, label="Autres classes")
for c, col in zip(top6, [BLUE, ORANGE, AQUA, YELLOW, MAGENTA, VIOLET]):
    m = (emb_sample["label"] == c).values
    ax.scatter(xy[m, 0], xy[m, 1], s=26, color=col, edgecolors=SURFACE, linewidths=0.6, label=short(c))
ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), fontsize=8)
ax.set_title(f"t-SNE des descripteurs ImageNet — k-NN (5 voisins) : {KNN_BASELINE:.1%}")
savefig("fig_tsne")

### 2.9 Bilan de l'EDA → décisions de prétraitement
Complétez la colonne « Constat » avec les valeurs affichées ci-dessus : c'est ce tableau qui relie l'EDA au
prétraitement dans le papier et à l'oral.

| Constat de l'EDA (à chiffrer) | Décision de prétraitement | Section |
|---|---|---|
| Fichiers illisibles | Suppression | 3.1 |
| Images minuscules ou très allongées | Filtrage (petit côté < 96 px, rapport > 2,5) | 3.1 |
| Doublons exacts et quasi-doublons | Une seule copie conservée, **avant** le découpage (pas de fuite) | 3.1 |
| Même photo dans deux classes | Suppression du groupe entier (étiquette douteuse) | 3.1 |
| Déséquilibre des classes | Découpage stratifié, perte pondérée, F1 macro | 3.2 |
| Résolutions hétérogènes, orientation EXIF | Cache 256 px normalisé | 2.2 |
| Luminosité et netteté variables | Augmentation photométrique + tests de robustesse | 3.3 et 5.7 |

## 3. Prétraitement
### 3.1 Nettoyage

In [ ]:
n0 = len(meta)
clean = ok.copy()
steps = [("Fichiers indexés", n0), ("Lisibles", len(clean))]
clean = clean[(clean[["width", "height"]].min(axis=1) >= 96) & clean["aspect"].between(0.4, 2.5)]
steps.append(("Taille et format acceptables", len(clean)))
clean = clean[clean["dup_nlabels"] == 1]
steps.append(("Sans conflit d'étiquette", len(clean)))
clean = clean[clean["dup_first"]]
steps.append(("Sans doublon", len(clean)))
clean = clean.reset_index(drop=True)

CLASS_NAMES = sorted(clean["label"].unique())
N_CLASSES = len(CLASS_NAMES)
clean["y"] = clean["label"].map({c: i for i, c in enumerate(CLASS_NAMES)}).astype("int32")
N_REMOVED = n0 - len(clean)

cleaning_log = pd.DataFrame(steps, columns=["Étape", "Images restantes"])
cleaning_log["Retirées"] = -cleaning_log["Images restantes"].diff().fillna(0).astype(int)
display(cleaning_log)
print(f"Jeu final : {len(clean):,} images · {N_CLASSES} classes · {N_REMOVED} retirées ({N_REMOVED / n0:.1%})")

### 3.2 Découpage stratifié 70 / 15 / 15 et poids de classes
- **Stratifié** : chaque sous-ensemble garde les proportions de classes (indispensable avec des classes rares).
- **Validation** : sert au réglage des hyperparamètres, à l'arrêt anticipé, à la calibration et aux seuils du jeu.
- **Test** : n'est consulté qu'une fois, en section 5. Aucune décision n'est prise dessus.
- **Poids de classes** (`balanced`) : une erreur sur une classe rare coûte plus cher, pour que le modèle ne la sacrifie pas.

In [ ]:
train_df, tmp_df = train_test_split(clean, test_size=0.30, stratify=clean["y"], random_state=SEED)
val_df, test_df = train_test_split(tmp_df, test_size=0.50, stratify=tmp_df["y"], random_state=SEED)
train_df, val_df, test_df = [d.reset_index(drop=True) for d in (train_df, val_df, test_df)]
assert not set(train_df["dhash"]) & set(test_df["dhash"]), "Fuite : un doublon est à la fois en train et en test"

class_w = compute_class_weight("balanced", classes=np.arange(N_CLASSES), y=train_df["y"].values).astype("float32")
y_val, y_test = val_df["y"].values, test_df["y"].values
print({k: len(v) for k, v in [("train", train_df), ("val", val_df), ("test", test_df)]})
print(f"Poids de classes : min {class_w.min():.2f} · max {class_w.max():.2f}")

### 3.3 Pipeline `tf.data` et augmentation
Chaque transformation simule une variation **réelle** d'un scan sur téléphone ; aucune ne change l'espèce :

| Augmentation | Ce qu'elle simule | Réglage |
|---|---|---|
| Recadrage aléatoire 224 dans 256 | Cadrage approximatif de l'utilisateur | — |
| Miroir horizontal | Plante vue de l'autre côté | p = 0,5 |
| Rotation, zoom | Téléphone incliné, distance variable | ±22°, −15 % à +10 % |
| Luminosité, contraste, saturation | Éclairage intérieur/extérieur, capteurs différents | ±15 %, ×0,8–1,25 |

Pas de miroir vertical (une plante en pot a un haut et un bas) ni de changement de teinte (la couleur distingue les espèces).
La normalisation des pixels est **dans le modèle** : l'application de démo lui envoie donc directement des pixels 0–255.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
CLASS_W = tf.constant(class_w)
aug_rotate = layers.RandomRotation(0.06, fill_mode="reflect")       # ±0,06 tour ≈ ±22°
aug_zoom = layers.RandomZoom((-0.15, 0.10), fill_mode="reflect")

def geo_aug(x):
    return aug_zoom(aug_rotate(x, training=True), training=True)

def decode(path, y):
    return tf.cast(tf.io.decode_jpeg(tf.io.read_file(path), channels=3), tf.float32), y

def photo_aug(img, y):
    img = tf.image.random_crop(img, [IMG_SIZE, IMG_SIZE, 3])
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, 0.15 * 255)
    img = tf.image.random_contrast(img, 0.8, 1.25)
    img = tf.image.random_saturation(img, 0.8, 1.25)
    return tf.ensure_shape(tf.clip_by_value(img, 0.0, 255.0), [IMG_SIZE, IMG_SIZE, 3]), y

def center_crop(img, y):
    return tf.ensure_shape(tf.image.resize_with_crop_or_pad(img, IMG_SIZE, IMG_SIZE), [IMG_SIZE, IMG_SIZE, 3]), y

def make_ds(frame, training=False, augment=True, batch=BATCH):
    ds = tf.data.Dataset.from_tensor_slices((frame["cache_path"].values, frame["y"].values.astype("int32")))
    if training:
        ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(decode, num_parallel_calls=AUTOTUNE)
    ds = ds.map(photo_aug if (training and augment) else center_crop, num_parallel_calls=AUTOTUNE).batch(batch)
    if training and augment:
        ds = ds.map(lambda x, y: (tf.clip_by_value(geo_aug(x), 0.0, 255.0), y), num_parallel_calls=AUTOTUNE)
    if training:   # étiquettes one-hot (lissage d'étiquettes) + poids par exemple
        ds = ds.map(lambda x, y: (x, tf.one_hot(y, N_CLASSES), tf.gather(CLASS_W, y)))
    else:
        ds = ds.map(lambda x, y: (x, tf.one_hot(y, N_CLASSES)))
    return ds.prefetch(AUTOTUNE)

train_ds, train_noaug_ds = make_ds(train_df, True, True), make_ds(train_df, True, False)
val_ds, test_ds = make_ds(val_df), make_ds(test_df)

In [ ]:
# Contrôle visuel : la même image, 11 augmentations. Les plantes doivent rester reconnaissables.
one = train_df.sample(1, random_state=SEED)
demo_ds = make_ds(pd.concat([one] * 11, ignore_index=True), True, True, batch=11)
aug_imgs = next(iter(demo_ds))[0].numpy().astype("uint8")
fig, axes = plt.subplots(2, 6, figsize=(12, 4.3))
axes[0, 0].imshow(Image.open(one["cache_path"].iloc[0])); axes[0, 0].set_title("Originale (256 px)", fontsize=9)
for ax, im in zip(axes.ravel()[1:], aug_imgs):
    ax.imshow(im)
for ax in axes.ravel():
    ax.axis("off")
fig.suptitle("Augmentation : une image, onze variantes", x=0.01, ha="left", fontweight="bold")
plt.tight_layout(); savefig("fig_augmentation")

## 4. Modélisation
### 4.1 Quatre architectures, un même protocole

| Modèle | Rôle dans l'étude | Justification |
|---|---|---|
| **CNN de référence** (entraîné de zéro) | Plancher | Mesure ce que vaut le transfert d'apprentissage sur un jeu de cette taille |
| **MobileNetV2** | Candidat mobile | Convolutions séparables et blocs résiduels inversés : conçu pour l'embarqué |
| **ResNet50V2** | Référence « lourde » | Connexions résiduelles, ≈ 7× plus de paramètres que MobileNetV2 : la capacité paie-t-elle ? |
| **EfficientNetB0** | Compromis | Mise à l'échelle composée profondeur/largeur/résolution |

**Transfert en deux phases.** (1) Base gelée, on entraîne seulement la tête : les poids ImageNet ne sont pas
détruits par les gradients d'une tête initialisée au hasard. (2) On dégèle les 30 % de couches les plus hautes
avec un taux d'apprentissage très faible : les couches basses (contours, textures) sont génériques, les couches
hautes se spécialisent sur les plantes. Les couches *BatchNormalization* restent gelées (leurs statistiques
ImageNet seraient dégradées par de petits lots).

La tête se termine par une couche `logits` puis un `softmax` séparé : Grad-CAM et la calibration ont besoin des logits.

In [ ]:
def conv_block(x, filters, name):
    for i in (1, 2):
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False, name=f"{name}_conv{i}")(x)
        x = layers.BatchNormalization(name=f"{name}_bn{i}")(x)
        x = layers.ReLU(name=f"{name}_relu{i}")(x)
    return x

def build_model(name, dropout=0.3, dense_units=0):
    """Construit un modèle « à plat » (sans sous-modèle imbriqué) pour que Grad-CAM accède aux cartes d'activation."""
    inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="image")          # pixels 0–255
    base = None
    if name == "cnn_scratch":
        x = layers.Rescaling(1 / 255.0, name="rescale")(inputs)
        for i, f in enumerate((32, 64, 128, 256)):
            x = conv_block(x, f, f"b{i + 1}")
            x = layers.MaxPooling2D(name=f"b{i + 1}_pool")(x)
        x = conv_block(x, 256, "b5")
    elif name == "mobilenetv2":
        x = layers.Rescaling(1 / 127.5, offset=-1, name="rescale")(inputs)   # [-1, 1]
        base = keras.applications.MobileNetV2(include_top=False, weights="imagenet", input_tensor=x)
    elif name == "resnet50v2":
        x = layers.Rescaling(1 / 127.5, offset=-1, name="rescale")(inputs)   # [-1, 1]
        base = keras.applications.ResNet50V2(include_top=False, weights="imagenet", input_tensor=x)
    elif name == "efficientnetb0":
        base = keras.applications.EfficientNetB0(include_top=False, weights="imagenet", input_tensor=inputs)  # normalise en interne
    else:
        raise ValueError(name)
    if base is not None:
        base.trainable = False
        x = base.output
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    if dense_units:
        x = layers.Dense(dense_units, activation="relu", name="head_dense")(x)
    x = layers.Dropout(dropout, name="head_dropout")(x)
    logits = layers.Dense(N_CLASSES, name="logits")(x)
    outputs = layers.Activation("softmax", name="probs")(logits)
    return keras.Model(inputs, outputs, name=name), base

def compile_model(model, lr, label_smoothing=0.05):
    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss=keras.losses.CategoricalCrossentropy(label_smoothing=label_smoothing),
                  metrics=["accuracy", keras.metrics.TopKCategoricalAccuracy(k=3, name="top3")])

def unfreeze_top(base, fraction=FT_FRACTION):
    """Dégèle la fraction haute de la base, sauf les BatchNormalization."""
    base.trainable = True
    cut = int(len(base.layers) * (1 - fraction))
    for i, layer in enumerate(base.layers):
        layer.trainable = (i >= cut) and not isinstance(layer, layers.BatchNormalization)

def count_params(model):
    return int(sum(np.prod(w.shape) for w in model.weights))

def fit(model, tr, va, epochs, patience=3, extra=()):
    cb = [keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=patience, restore_best_weights=True), *extra]
    return model.fit(tr, validation_data=va, epochs=epochs, callbacks=cb, verbose=2).history

def train_model(name, hp, tr=None, va=None, epochs_head=None, epochs_ft=None):
    """Entraîne un modèle selon le protocole commun. Renvoie (modèle, historique, époque de début du fine-tuning, durée)."""
    tr = train_ds if tr is None else tr
    va = val_ds if va is None else va
    epochs_head = EPOCHS_HEAD if epochs_head is None else epochs_head
    epochs_ft = EPOCHS_FT if epochs_ft is None else epochs_ft
    set_seed()
    model, base = build_model(name, hp["dropout"], hp["dense_units"])
    t0 = time.time()
    if base is None:                                    # CNN de zéro : une seule phase, plus longue
        compile_model(model, 1e-3, hp["label_smoothing"])
        hist = fit(model, tr, va, EPOCHS_CNN, patience=5,
                   extra=[keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2)])
        ft_start = None
    else:
        compile_model(model, hp["lr"], hp["label_smoothing"])
        hist = fit(model, tr, va, epochs_head)
        ft_start = len(hist["loss"])
        if epochs_ft > 0:
            unfreeze_top(base)
            compile_model(model, LR_FT, hp["label_smoothing"])   # recompiler après avoir changé `trainable`
            h2 = fit(model, tr, va, epochs_ft)
            hist = {k: hist[k] + h2[k] for k in hist if k in h2}
    return model, hist, ft_start, time.time() - t0

DEFAULT_HP = {"lr": 1e-3, "dropout": 0.3, "dense_units": 0, "label_smoothing": 0.05}
for n in MODEL_NAMES:
    m, _ = build_model(n)
    print(f"{MODEL_LABEL[n]:<18} {count_params(m) / 1e6:6.2f} M paramètres")
    del m
keras.backend.clear_session(); gc.collect();

### 4.2 Ablation : l'augmentation sert-elle à quelque chose ?
Même modèle (MobileNetV2, tête seule), mêmes hyperparamètres, mêmes époques : seul le pipeline d'entrée change.
Les deux modèles sont sauvegardés et comparés en section 5.7 sur les images dégradées — c'est là que
l'augmentation doit faire la différence.

In [ ]:
ablation = {}
for tag, ds in [("avec_aug", train_ds), ("sans_aug", train_noaug_ds)]:
    m, h, _, _ = train_model("mobilenetv2", DEFAULT_HP, tr=ds, epochs_ft=0)
    m.save(str(MODEL_DIR / f"ablation_{tag}.keras"))
    ablation[tag] = {"val_acc": max(h["val_accuracy"]), "train_acc": h["accuracy"][-1]}
    del m; keras.backend.clear_session(); gc.collect()
ablation_df = pd.DataFrame(ablation).T
ablation_df["écart train − val"] = ablation_df["train_acc"] - ablation_df["val_acc"]
display(ablation_df.round(4))

### 4.3 Recherche d'hyperparamètres (Optuna)
- **Espace** : taux d'apprentissage (échelle log), dropout, taille d'une couche dense optionnelle, lissage d'étiquettes.
- **Algorithme** : TPE (recherche bayésienne) plutôt qu'une grille : à budget égal, il concentre les essais dans les zones prometteuses.
- **Élagage** (*median pruner*) : un essai nettement sous la médiane est arrêté tôt.
- **Budget** : la recherche est faite sur MobileNetV2 (le plus rapide), base gelée, puis les hyperparamètres de tête
  sont réutilisés pour les autres bases. C'est une **limite assumée**, imposée par le GPU gratuit ; elle est signalée dans le papier.
- L'objectif est l'accuracy de **validation** ; le jeu de test n'intervient jamais ici.

In [ ]:
try:
    import optuna
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "optuna"], check=True)
    import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    hp = {"lr": trial.suggest_float("lr", 1e-4, 3e-3, log=True),
          "dropout": trial.suggest_float("dropout", 0.1, 0.5),
          "dense_units": trial.suggest_categorical("dense_units", [0, 128, 256]),
          "label_smoothing": trial.suggest_float("label_smoothing", 0.0, 0.15)}
    keras.backend.clear_session(); gc.collect(); set_seed()
    model, _ = build_model("mobilenetv2", hp["dropout"], hp["dense_units"])
    compile_model(model, hp["lr"], hp["label_smoothing"])

    class Prune(keras.callbacks.Callback):
        def on_epoch_end(self, epoch, logs=None):
            trial.report(logs["val_accuracy"], epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

    hist = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_TRIAL, callbacks=[Prune()], verbose=0).history
    return max(hist["val_accuracy"])

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED),
                            pruner=optuna.pruners.MedianPruner(n_startup_trials=4, n_warmup_steps=1))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)
BEST_HP = dict(study.best_params)
print(f"Meilleure accuracy de validation : {study.best_value:.4f}\nHyperparamètres retenus : {BEST_HP}")
keras.backend.clear_session(); gc.collect();

In [ ]:
trials = study.trials_dataframe()
done = trials[trials["state"] == "COMPLETE"]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
axes[0].scatter(trials["number"], trials["value"], color=BLUE, s=36, edgecolors=SURFACE, linewidths=0.8)
axes[0].plot(done["number"], done["value"].cummax(), color=INK, lw=1.2, drawstyle="steps-post")
axes[0].set_xlabel("Essai"); axes[0].set_ylabel("Accuracy de validation")
axes[0].set_title("Historique (trait : meilleur score atteint)")
axes[1].scatter(done["params_lr"], done["value"], color=BLUE, s=36, edgecolors=SURFACE, linewidths=0.8)
axes[1].set_xscale("log"); axes[1].set_xlabel("Taux d'apprentissage"); axes[1].set_title("Sensibilité au taux d'apprentissage")
try:
    imp = optuna.importance.get_param_importances(study)
    axes[2].barh(list(imp)[::-1], list(imp.values())[::-1], color=BLUE, height=0.6)
    axes[2].set_xlabel("Importance relative"); axes[2].grid(axis="y", visible=False)
except Exception as e:
    axes[2].text(0.5, 0.5, "Importance indisponible\n(trop peu d'essais)", ha="center", va="center", color=MUTED); axes[2].axis("off")
axes[2].set_title("Importance des hyperparamètres")
plt.tight_layout(); savefig("fig_optuna")
print(f"{len(done)} essais terminés · {int((trials['state'] == 'PRUNED').sum())} élagués")

### 4.4 Entraînement final des quatre modèles
Si un modèle échoue (téléchargement des poids, mémoire), le notebook continue avec les autres et le signale.

In [ ]:
models, histories, ft_starts, train_times = {}, {}, {}, {}
for name in MODEL_NAMES:
    print(f"\n{'=' * 22} {MODEL_LABEL[name]} {'=' * 22}")
    try:
        m, h, fs, dt = train_model(name, BEST_HP)
        m.save(str(MODEL_DIR / f"{name}.keras"))
        models[name], histories[name], ft_starts[name], train_times[name] = m, h, fs, dt
        print(f"→ meilleure accuracy de validation : {max(h['val_accuracy']):.4f} · {dt / 60:.1f} min")
    except Exception as e:
        print(f"⚠️ {name} ignoré : {type(e).__name__}: {e}")
TRAINED = list(models)
json.dump({n: {k: [float(v) for v in vals] for k, vals in h.items()} for n, h in histories.items()},
          open(WORK / "histories.json", "w"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharex=True)
for name in TRAINED:
    h = histories[name]; ep = np.arange(1, len(h["val_accuracy"]) + 1)
    for ax, key in zip(axes, ["val_accuracy", "val_loss"]):
        ax.plot(ep, h[key], color=MODEL_COLOR[name], label=MODEL_LABEL[name])
        if ft_starts[name] and ft_starts[name] < len(ep):
            ax.plot(ft_starts[name] + 1, h[key][ft_starts[name]], "o", color=MODEL_COLOR[name],
                    markeredgecolor=SURFACE, markeredgewidth=1.5, markersize=8)
axes[0].set_title("Accuracy de validation (● : début du fine-tuning)"); axes[1].set_title("Perte de validation")
for ax in axes:
    ax.set_xlabel("Époque")
axes[1].legend(loc="upper right")
plt.tight_layout(); savefig("fig_learning_curves")

## 5. Évaluation et interprétation
### 5.1 Prédictions sur validation et test
Les logits sont calculés une fois et réutilisés partout. **C'est la première et unique fois que le jeu de test est utilisé.**

In [ ]:
def predict_logits(model, ds):
    logit_model = keras.Model(model.input, model.get_layer("logits").output)
    return logit_model.predict(ds, verbose=0).astype("float64")

def gpu_latency_ms(model, batch=32, reps=10):
    """Temps moyen par image (ms) en lot de 32, sur le matériel courant."""
    x = tf.constant(np.random.rand(batch, IMG_SIZE, IMG_SIZE, 3).astype("float32") * 255)
    fn = tf.function(lambda t: model(t, training=False))
    fn(x)                                                   # compilation + échauffement
    t0 = time.perf_counter()
    for _ in range(reps):
        fn(x).numpy()
    return (time.perf_counter() - t0) / (reps * batch) * 1000

P, static = {}, {}
for name in TRAINED:
    P[name] = {"val": predict_logits(models[name], val_ds), "test": predict_logits(models[name], test_ds)}
    static[name] = {"params_M": count_params(models[name]) / 1e6,
                    "keras_MB": (MODEL_DIR / f"{name}.keras").stat().st_size / 1e6,
                    "gpu_ms": gpu_latency_ms(models[name]), "train_min": train_times[name] / 60}
np.savez_compressed(WORK / "logits.npz", y_val=y_val, y_test=y_test,
                    **{f"{n}_{s}": P[n][s] for n in P for s in ("val", "test")})

### 5.2 Tableau comparatif
- **Accuracy** : part de bonnes réponses — avec son intervalle de confiance à 95 % (bootstrap, 1 000 rééchantillonnages).
- **Accuracy équilibrée** et **F1 macro** : chaque classe pèse autant, quelle que soit sa fréquence.
- **Top-3** : la bonne espèce est parmi les 3 propositions — c'est ce que voit le joueur dans le mini-quiz.
- **AUC macro** (un-contre-tous) : qualité du classement des scores, indépendante du seuil.

In [ ]:
def bootstrap_ci(correct, n_boot=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    means = correct[rng.integers(0, len(correct), (n_boot, len(correct)))].mean(axis=1)
    return np.percentile(means, [2.5, 97.5])

def evaluate(logits, y):
    probs = softmax(logits, axis=1); pred = probs.argmax(1)
    lo, hi = bootstrap_ci((pred == y).astype(float))
    try:
        auc_macro = roc_auc_score(y, probs, multi_class="ovr", average="macro", labels=np.arange(probs.shape[1]))
    except ValueError:                                  # une classe absente du jeu évalué
        auc_macro = float("nan")
    return {"accuracy": accuracy_score(y, pred), "ci_low": lo, "ci_high": hi,
            "balanced_acc": balanced_accuracy_score(y, pred),
            "f1_macro": f1_score(y, pred, average="macro", zero_division=0),
            "top3": top_k_accuracy_score(y, probs, k=3, labels=np.arange(probs.shape[1])),
            "auc_macro": auc_macro}

results = pd.DataFrame({n: {**evaluate(P[n]["test"], y_test), **static[n]} for n in TRAINED}).T
results = results.sort_values("f1_macro", ascending=False)
BEST = results.index[0]
RUNNER_UP = results.index[1] if len(results) > 1 else BEST
display(results.rename(index=MODEL_LABEL).style.format(
    {"accuracy": "{:.2%}", "ci_low": "{:.2%}", "ci_high": "{:.2%}", "balanced_acc": "{:.2%}", "f1_macro": "{:.2%}",
     "top3": "{:.2%}", "auc_macro": "{:.4f}", "params_M": "{:.2f}", "keras_MB": "{:.1f}", "gpu_ms": "{:.2f}",
     "train_min": "{:.1f}"}).highlight_max(subset=["accuracy", "balanced_acc", "f1_macro", "top3", "auc_macro"], color="#cfe3fa"))
print(f"Meilleur modèle (F1 macro) : {MODEL_LABEL[BEST]} · plancher k-NN sans entraînement : {globals().get('KNN_BASELINE', float('nan')):.2%}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.3), sharey=True)
order = results.index[::-1]
for ax, (col, title) in zip(axes, [("accuracy", "Accuracy (IC 95 %)"), ("f1_macro", "F1 macro"), ("top3", "Top-3")]):
    vals = results.loc[order, col].values
    ax.barh([MODEL_LABEL[n] for n in order], vals, color=[MODEL_COLOR[n] for n in order], height=0.6)
    if col == "accuracy":
        ax.errorbar(vals, range(len(order)), xerr=[vals - results.loc[order, "ci_low"].values,
                    results.loc[order, "ci_high"].values - vals], fmt="none", ecolor=INK, elinewidth=1, capsize=3)
    for i, v in enumerate(vals):
        ax.text(0.02, i, f"{v:.1%}", va="center", color="white", fontweight="bold", fontsize=9)
    ax.set_xlim(0, 1.02); ax.set_title(title); ax.grid(axis="y", visible=False)
    ax.xaxis.set_major_formatter(mpl.ticker.PercentFormatter(1, decimals=0))
plt.tight_layout(); savefig("fig_model_comparison")

### 5.3 L'écart entre les deux meilleurs modèles est-il significatif ? (test de McNemar)
Les deux modèles sont évalués sur les **mêmes** images : on compare les cas où l'un a raison et l'autre tort.
Si p > 0,05, on ne peut pas affirmer que l'un est meilleur — le choix se fera alors sur la latence et la taille.

In [ ]:
def mcnemar(logits_a, logits_b, y):
    a, b = logits_a.argmax(1) == y, logits_b.argmax(1) == y
    n_ab, n_ba = int((a & ~b).sum()), int((~a & b).sum())
    p = binomtest(min(n_ab, n_ba), n_ab + n_ba, 0.5).pvalue if (n_ab + n_ba) else 1.0
    return n_ab, n_ba, p

n_ab, n_ba, MCNEMAR_P = mcnemar(P[BEST]["test"], P[RUNNER_UP]["test"], y_test)
print(f"{MODEL_LABEL[BEST]} seul correct : {n_ab} images · {MODEL_LABEL[RUNNER_UP]} seul correct : {n_ba} images")
print(f"McNemar exact : p = {MCNEMAR_P:.4f} → écart", "significatif" if MCNEMAR_P < 0.05 else "NON significatif", "au seuil de 5 %")

### 5.4 Matrice de confusion, performances par classe, courbes ROC

In [ ]:
probs_best = softmax(P[BEST]["test"], axis=1)
pred_best = probs_best.argmax(1)
cm = confusion_matrix(y_test, pred_best, labels=np.arange(N_CLASSES))
cm_norm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)

fig, ax = plt.subplots(figsize=(11, 9.5))
sns.heatmap(cm_norm, cmap=SEQ_CMAP, vmin=0, vmax=1, square=True, ax=ax, linewidths=0.3, linecolor=SURFACE,
            xticklabels=[short(c, 16) for c in CLASS_NAMES], yticklabels=[short(c, 16) for c in CLASS_NAMES],
            cbar_kws={"shrink": 0.6, "label": "Part des images de la classe réelle"})
ax.set_xlabel("Classe prédite"); ax.set_ylabel("Classe réelle"); ax.tick_params(labelsize=6.5)
ax.set_title(f"Matrice de confusion normalisée — {MODEL_LABEL[BEST]} (test)")
savefig("fig_confusion")

off = cm.copy(); np.fill_diagonal(off, 0)
pairs = (pd.DataFrame([(CLASS_NAMES[i], CLASS_NAMES[j], off[i, j], cm_norm[i, j])
                       for i in range(N_CLASSES) for j in range(N_CLASSES) if off[i, j] > 0],
                      columns=["Classe réelle", "Prédite comme", "Erreurs", "Part de la classe"])
         .sort_values("Erreurs", ascending=False).head(10).reset_index(drop=True))
print("Les 10 confusions les plus fréquentes :"); display(pairs.style.format({"Part de la classe": "{:.1%}"}))

In [ ]:
report = pd.DataFrame(classification_report(y_test, pred_best, labels=np.arange(N_CLASSES), target_names=CLASS_NAMES,
                                            output_dict=True, zero_division=0)).T.iloc[:N_CLASSES]
report["n_train"] = train_df["label"].value_counts().reindex(CLASS_NAMES).values
rep = report.sort_values("f1-score")

fig, axes = plt.subplots(1, 2, figsize=(13, max(4.5, 0.2 * N_CLASSES)), gridspec_kw={"width_ratios": [1.5, 1]})
axes[0].barh([short(c) for c in rep.index], rep["f1-score"], color=BLUE, height=0.72)
axes[0].axvline(report["f1-score"].mean(), color=MUTED, lw=1, ls="--")
axes[0].set_xlim(0, 1); axes[0].set_xlabel("F1"); axes[0].grid(axis="y", visible=False); axes[0].tick_params(axis="y", labelsize=7.5)
axes[0].set_title("F1 par classe (pointillé : moyenne macro)")
axes[1].scatter(report["n_train"], report["f1-score"], color=BLUE, s=40, edgecolors=SURFACE, linewidths=0.8)
RHO_SUPPORT = report[["n_train", "f1-score"]].corr(method="spearman").iloc[0, 1]
axes[1].set_xlabel("Images d'entraînement de la classe"); axes[1].set_ylabel("F1")
axes[1].set_title("Effet du déséquilibre (classes de même effectif)" if np.isnan(RHO_SUPPORT)
                  else f"Effet du déséquilibre (ρ de Spearman = {RHO_SUPPORT:.2f})")
plt.tight_layout(); savefig("fig_per_class_f1")

In [ ]:
Y_bin = label_binarize(y_test, classes=np.arange(N_CLASSES))
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for name in TRAINED:
    pr = softmax(P[name]["test"], axis=1)
    fpr, tpr, _ = roc_curve(Y_bin.ravel(), pr.ravel())
    axes[0].plot(fpr, tpr, color=MODEL_COLOR[name], label=f"{MODEL_LABEL[name]} (AUC = {auc(fpr, tpr):.4f})")
axes[0].plot([0, 1], [0, 1], color=GRID, lw=1); axes[0].set_xscale("log"); axes[0].set_xlim(1e-4, 1)
axes[0].set_xlabel("Taux de faux positifs (échelle log)"); axes[0].set_ylabel("Taux de vrais positifs")
axes[0].set_title("ROC micro-moyennée (un-contre-tous)"); axes[0].legend(loc="upper left", fontsize=8.5)

class_auc = pd.Series({c: auc(*roc_curve(Y_bin[:, i], probs_best[:, i])[:2]) for i, c in enumerate(CLASS_NAMES)
                       if Y_bin[:, i].any()}).sort_values()
worst = class_auc.head(8)
axes[1].barh([short(c) for c in worst.index[::-1]], worst.values[::-1], color=BLUE, height=0.6)
axes[1].set_xlim(min(0.5, worst.min() - 0.02), 1); axes[1].set_xlabel("AUC"); axes[1].grid(axis="y", visible=False)
axes[1].set_title(f"AUC les plus faibles — {MODEL_LABEL[BEST]}")
plt.tight_layout(); savefig("fig_roc")

### 5.5 Calibration et seuils du jeu
Le jeu décide selon la **confiance** du modèle : encore faut-il qu'un « 90 % » signifie vraiment 90 % de chances
d'avoir raison. On mesure l'erreur de calibration (ECE), on la corrige par **mise à l'échelle en température**
(un seul paramètre T, ajusté sur la validation), puis on fixe deux seuils **sur la validation** :

| Confiance | Mécanique de jeu | Règle de choix du seuil |
|---|---|---|
| ≥ `T_HIGH` | 🃏 Carte débloquée | plus petit seuil tel que le top-1 soit juste ≥ 95 % du temps |
| entre les deux | ❓ Mini-quiz sur le top-3 | plus petit seuil tel que le top-3 contienne la bonne espèce ≥ 95 % du temps |
| < `T_LOW` | 🔄 « Rescanne sous un autre angle » | — |

Les taux obtenus sont ensuite **vérifiés sur le test**.

In [ ]:
def ece_score(probs, y, n_bins=15):
    conf, correct = probs.max(1), probs.argmax(1) == y
    edges = np.linspace(0, 1, n_bins + 1); ece, rows = 0.0, []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
            rows.append((conf[m].mean(), correct[m].mean(), int(m.sum())))
    return ece, np.array(rows)

def fit_temperature(logits, y):
    nll = lambda t: -log_softmax(logits / t, axis=1)[np.arange(len(y)), y].mean()
    return float(minimize_scalar(nll, bounds=(0.05, 10), method="bounded").x)

def threshold_for(conf, hit, target=0.95):
    """Plus petit seuil de confiance tel que le taux de réussite des prédictions acceptées soit ≥ target."""
    order = np.argsort(-conf)
    running = np.cumsum(hit[order]) / np.arange(1, len(conf) + 1)
    ok_idx = np.where(running >= target)[0]
    return float(conf[order][ok_idx.max()]) if len(ok_idx) else 1.0

def top3_hit(probs, y):
    return (np.argsort(-probs, axis=1)[:, :3] == y[:, None]).any(axis=1)

TEMP = fit_temperature(P[BEST]["val"], y_val)
val_cal, test_cal = softmax(P[BEST]["val"] / TEMP, axis=1), softmax(P[BEST]["test"] / TEMP, axis=1)
ECE_BEFORE, rel_before = ece_score(probs_best, y_test)
ECE_AFTER, rel_after = ece_score(test_cal, y_test)

T_HIGH = threshold_for(val_cal.max(1), val_cal.argmax(1) == y_val)
T_LOW = min(T_HIGH, threshold_for(val_cal.max(1), top3_hit(val_cal, y_val)))
conf_t, hit1_t, hit3_t = test_cal.max(1), test_cal.argmax(1) == y_test, top3_hit(test_cal, y_test)
zone_hi, zone_lo = conf_t >= T_HIGH, conf_t < T_LOW
zone_mid = ~zone_hi & ~zone_lo
GAME = {"T": TEMP, "t_high": T_HIGH, "t_low": T_LOW,
        "share_card": zone_hi.mean(), "acc_card": hit1_t[zone_hi].mean() if zone_hi.any() else float("nan"),
        "share_quiz": zone_mid.mean(), "top3_quiz": hit3_t[zone_mid].mean() if zone_mid.any() else float("nan"),
        "share_rescan": zone_lo.mean()}
print(f"Température T = {TEMP:.3f} · ECE {ECE_BEFORE:.4f} → {ECE_AFTER:.4f}")
print(f"Seuils (validation) : carte ≥ {T_HIGH:.3f} · quiz ≥ {T_LOW:.3f}")
print(f"Sur le test : {GAME['share_card']:.1%} des scans débloquent une carte (justes à {GAME['acc_card']:.1%}) · "
      f"{GAME['share_quiz']:.1%} déclenchent un quiz (bonne réponse dans le top-3 : {GAME['top3_quiz']:.1%}) · "
      f"{GAME['share_rescan']:.1%} demandent un nouveau scan")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].plot([0, 1], [0, 1], color=GRID, lw=1.2)
for rows_, col, lab in [(rel_before, ORANGE, f"Avant (ECE = {ECE_BEFORE:.3f})"), (rel_after, BLUE, f"Après température (ECE = {ECE_AFTER:.3f})")]:
    axes[0].plot(rows_[:, 0], rows_[:, 1], "o-", color=col, label=lab, markeredgecolor=SURFACE)
axes[0].set_xlabel("Confiance annoncée"); axes[0].set_ylabel("Accuracy observée")
axes[0].set_title(f"Diagramme de fiabilité — {MODEL_LABEL[BEST]}"); axes[0].legend(loc="upper left")

order_ = np.argsort(-conf_t); cov = np.arange(1, len(conf_t) + 1) / len(conf_t)
axes[1].plot(cov, np.cumsum(hit1_t[order_]) / np.arange(1, len(conf_t) + 1), color=BLUE, label="Top-1")
axes[1].plot(cov, np.cumsum(hit3_t[order_]) / np.arange(1, len(conf_t) + 1), color=AQUA, label="Top-3")
for share, lab in [(zone_hi.mean(), "carte"), ((~zone_lo).mean(), "quiz")]:
    axes[1].axvline(share, color=MUTED, lw=1, ls="--")
    axes[1].text(share, axes[1].get_ylim()[0], f" seuil {lab}", rotation=90, va="bottom", ha="right", color=MUTED, fontsize=8.5)
axes[1].set_xlabel("Couverture : part des scans acceptés (du plus sûr au moins sûr)"); axes[1].set_ylabel("Accuracy des scans acceptés")
axes[1].set_title("Précision sélective selon la confiance"); axes[1].legend(loc="upper right")
plt.tight_layout(); savefig("fig_calibration")

### 5.6 Explicabilité (Grad-CAM) et analyse des erreurs
Grad-CAM pondère les cartes d'activation de la dernière couche convolutive par le gradient du score de la classe :
les zones chaudes sont celles qui ont **fait pencher la décision**. Ce qu'on vérifie : le modèle regarde-t-il la
plante (feuilles, fleurs) ou un **raccourci** (pot, arrière-plan, texte incrusté) ?

In [ ]:
def gradcam(model, images, class_idx=None):
    """Cartes Grad-CAM (N, h, w) normalisées dans [0, 1] pour un lot d'images 0–255."""
    grad_model = keras.Model(model.input, [model.get_layer("gap").input, model.get_layer("logits").output])
    x = tf.convert_to_tensor(images, tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(x)
        fmap, logits = grad_model(x, training=False)
        idx = tf.argmax(logits, axis=1) if class_idx is None else tf.convert_to_tensor(class_idx, tf.int64)
        score = tf.gather(logits, idx, axis=1, batch_dims=1)
    grads = tape.gradient(score, fmap)
    weights = tf.reduce_mean(grads, axis=(1, 2), keepdims=True)
    cam = tf.nn.relu(tf.reduce_sum(weights * fmap, axis=-1))
    cam = cam / (tf.reduce_max(cam, axis=(1, 2), keepdims=True) + 1e-8)
    return cam.numpy()

def overlay(img, cam, alpha=0.65):
    """Superpose la carte de chaleur : transparente là où le modèle ne regarde pas, rouge là où il regarde le plus."""
    heat = np.asarray(Image.fromarray(np.uint8(cam * 255)).resize((img.shape[1], img.shape[0]), Image.BICUBIC),
                      dtype=np.float32) / 255.0
    color = mpl.colormaps["turbo"](heat)[..., :3] * 255
    a = (alpha * np.clip(heat * 1.6, 0, 1))[..., None]
    return np.uint8(np.clip(img * (1 - a) + color * a, 0, 255))

test_df = test_df.assign(pred=pred_best, conf=test_cal.max(1), correct=pred_best == y_test)
good = test_df[test_df["correct"]].sample(min(6, int(test_df["correct"].sum())), random_state=SEED)
imgs = load_batch(good["cache_path"].values)
cams = gradcam(models[BEST], imgs)
fig, axes = plt.subplots(2, len(good), figsize=(2.1 * len(good), 4.6), squeeze=False)
for k, (_, r) in enumerate(good.iterrows()):
    axes[0, k].imshow(imgs[k].astype("uint8"))
    axes[0, k].set_title(f"{short(r['label'], 18)}\nconfiance {r['conf']:.0%}", fontsize=8, loc="center", fontweight="normal")
    axes[1, k].imshow(overlay(imgs[k], cams[k]))
for ax in axes.ravel():
    ax.axis("off")
fig.suptitle(f"Grad-CAM — prédictions correctes ({MODEL_LABEL[BEST]})", x=0.01, ha="left", fontweight="bold")
plt.tight_layout(); savefig("fig_gradcam")

In [ ]:
# Même image, quatre modèles : regardent-ils au même endroit ?
cmp_rows = test_df.sample(min(4, len(test_df)), random_state=SEED + 1)
cmp_imgs = load_batch(cmp_rows["cache_path"].values)
fig, axes = plt.subplots(len(cmp_rows), len(TRAINED) + 1, figsize=(2.1 * (len(TRAINED) + 1), 2.2 * len(cmp_rows)), squeeze=False)
for i, (_, r) in enumerate(cmp_rows.iterrows()):
    axes[i, 0].imshow(cmp_imgs[i].astype("uint8"))
    axes[i, 0].set_title(short(r["label"], 18), fontsize=8, loc="center", fontweight="normal")
for j, name in enumerate(TRAINED, start=1):
    cams_j = gradcam(models[name], cmp_imgs)
    pred_j = softmax(predict_logits(models[name], cmp_imgs), axis=1).argmax(1)
    for i in range(len(cmp_rows)):
        axes[i, j].imshow(overlay(cmp_imgs[i], cams_j[i]))
        right = pred_j[i] == cmp_rows["y"].values[i]
        axes[i, j].set_title(f"{MODEL_LABEL[name]} {'✓' if right else '✗'}", fontsize=8, loc="center", fontweight="normal")
for ax in axes.ravel():
    ax.axis("off")
fig.suptitle("Grad-CAM : comparaison des modèles", x=0.01, ha="left", fontweight="bold")
plt.tight_layout(); savefig("fig_gradcam_models")

In [ ]:
# Erreurs les plus confiantes : où regarde le modèle pour la classe prédite, et pour la vraie classe ?
errors = test_df[~test_df["correct"]].sort_values("conf", ascending=False)
top_err = errors.head(5)
if len(top_err):
    e_imgs = load_batch(top_err["cache_path"].values)
    cam_pred = gradcam(models[BEST], e_imgs, top_err["pred"].values)
    cam_true = gradcam(models[BEST], e_imgs, top_err["y"].values)
    fig, axes = plt.subplots(3, len(top_err), figsize=(2.3 * len(top_err), 7), squeeze=False)
    for k, (_, r) in enumerate(top_err.iterrows()):
        axes[0, k].imshow(e_imgs[k].astype("uint8")); axes[0, k].set_title(f"Réel : {short(r['label'], 16)}", fontsize=8, loc="center", fontweight="normal")
        axes[1, k].imshow(overlay(e_imgs[k], cam_pred[k])); axes[1, k].set_title(f"Prédit : {short(CLASS_NAMES[r['pred']], 14)} ({r['conf']:.0%})", fontsize=8, loc="center", fontweight="normal")
        axes[2, k].imshow(overlay(e_imgs[k], cam_true[k])); axes[2, k].set_title("Zones de la vraie classe", fontsize=8, loc="center", fontweight="normal", color=MUTED)
    for ax in axes.ravel():
        ax.axis("off")
    fig.suptitle("Les erreurs les plus confiantes", x=0.01, ha="left", fontweight="bold")
    plt.tight_layout(); plt.subplots_adjust(hspace=0.22); savefig("fig_errors")
print(f"{len(errors)} erreurs sur {len(test_df)} images de test ({len(errors) / len(test_df):.1%})")

**Les erreurs dépendent-elles de la qualité de l'image ?** On croise les erreurs avec les mesures de l'EDA
(section 2.6), par quartile. Un taux d'erreur nettement plus élevé dans le quartile le plus sombre ou le plus
flou confirme que la robustesse photométrique est un enjeu réel, pas seulement théorique.

**Grille de lecture des erreurs (à remplir en regardant les figures)** : (a) espèces réellement proches
(même genre, même famille) ; (b) étiquette douteuse dans le dataset ; (c) plante peu visible, plusieurs plantes ;
(d) raccourci visuel (pot, fond, texte).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.3), sharey=True)
for ax, col, lab in zip(axes, ["brightness", "sharpness", "contrast"], ["Luminosité", "Netteté", "Contraste"]):
    q = pd.qcut(test_df[col], 4, labels=["Q1 (bas)", "Q2", "Q3", "Q4 (haut)"], duplicates="drop")
    err = 1 - test_df.groupby(q, observed=True)["correct"].mean()
    ax.bar(err.index.astype(str), err.values, color=BLUE, width=0.6)
    for i, v in enumerate(err.values):
        ax.text(i, v, f"{v:.1%}", ha="center", va="bottom", fontsize=9)
    ax.set_title(f"Erreur selon : {lab.lower()}"); ax.grid(axis="x", visible=False)
    ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1, decimals=0))
plt.tight_layout(); savefig("fig_error_vs_quality")

### 5.7 Robustesse aux conditions réelles
En AR, l'image vient d'un téléphone tenu à la main : éclairage médiocre, flou de bougé, inclinaison, bruit du
capteur. On dégrade les images de test selon 5 perturbations × 5 niveaux de sévérité et on mesure la chute
d'accuracy de chaque modèle. Les deux modèles de l'ablation (avec / sans augmentation) sont testés aussi.

*Limite* : au-delà de 20°, la rotation fait apparaître des coins noirs que la caméra réelle ne produirait pas.

In [ ]:
SEVERITIES = [1, 2, 3, 4, 5]
CORRUPTIONS = {
    "Sous-exposition": lambda im, s: ImageEnhance.Brightness(im).enhance([0.8, 0.65, 0.5, 0.35, 0.2][s - 1]),
    "Surexposition":   lambda im, s: ImageEnhance.Brightness(im).enhance([1.2, 1.4, 1.6, 1.8, 2.0][s - 1]),
    "Flou":            lambda im, s: im.filter(ImageFilter.GaussianBlur([0.5, 1.0, 1.5, 2.5, 4.0][s - 1])),
    "Rotation":        lambda im, s: im.rotate([5, 10, 15, 20, 30][s - 1], resample=Image.BILINEAR),
    "Bruit":           lambda im, s: Image.fromarray(np.uint8(np.clip(
        np.asarray(im, dtype=np.float32) + np.random.default_rng(SEED).normal(0, [5, 10, 20, 35, 50][s - 1], (CACHE_SIZE, CACHE_SIZE, 3)), 0, 255))),
}

def corrupted_batch(paths, kind, s):
    o = (CACHE_SIZE - IMG_SIZE) // 2
    return np.stack([np.asarray(CORRUPTIONS[kind](Image.open(p).convert("RGB"), s), dtype=np.float32)[o:o + IMG_SIZE, o:o + IMG_SIZE]
                     for p in paths])

rob_df = per_class(test_df, max(1, N_ROBUST // N_CLASSES), col="y")
rob_y = rob_df["y"].values
rob_models = dict(models)
for tag in ("avec_aug", "sans_aug"):
    f = MODEL_DIR / f"ablation_{tag}.keras"
    if f.exists():
        rob_models[tag] = keras.models.load_model(str(f))

acc_on = lambda m, x: float((m.predict(x, batch_size=64, verbose=0).argmax(1) == rob_y).mean())
clean_x = load_batch(rob_df["cache_path"].values)
rob_rows = [{"model": n, "corruption": "Aucune", "severity": 0, "accuracy": acc_on(m, clean_x)} for n, m in rob_models.items()]
for kind in CORRUPTIONS:
    for s in SEVERITIES:
        x = corrupted_batch(rob_df["cache_path"].values, kind, s)
        rob_rows += [{"model": n, "corruption": kind, "severity": s, "accuracy": acc_on(m, x)} for n, m in rob_models.items()]
rob = pd.DataFrame(rob_rows)
rob.to_csv(WORK / "robustness.csv", index=False)

In [ ]:
# Aperçu des perturbations (sévérités 1, 3, 5)
p0 = rob_df["cache_path"].iloc[0]
fig, axes = plt.subplots(3, len(CORRUPTIONS), figsize=(2.1 * len(CORRUPTIONS), 6.3))
for j, kind in enumerate(CORRUPTIONS):
    for i, s in enumerate((1, 3, 5)):
        axes[i, j].imshow(corrupted_batch([p0], kind, s)[0].astype("uint8")); axes[i, j].axis("off")
        axes[i, j].set_title(f"{kind} · niveau {s}", fontsize=8, loc="center", fontweight="normal")
plt.tight_layout(); savefig("fig_corruptions_preview")

fig, axes = plt.subplots(1, len(CORRUPTIONS), figsize=(3 * len(CORRUPTIONS), 3.5), sharey=True)
for ax, kind in zip(axes, CORRUPTIONS):
    for name in TRAINED:
        base_acc = rob[(rob["model"] == name) & (rob["severity"] == 0)]["accuracy"].iloc[0]
        d = rob[(rob["model"] == name) & (rob["corruption"] == kind)].sort_values("severity")
        ax.plot([0, *d["severity"]], [base_acc, *d["accuracy"]], "o-", color=MODEL_COLOR[name],
                label=MODEL_LABEL[name], markeredgecolor=SURFACE, markersize=5)
    ax.set_title(kind); ax.set_xlabel("Sévérité"); ax.set_xticks([0, *SEVERITIES])
    ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1, decimals=0))
axes[0].set_ylabel("Accuracy"); axes[-1].legend(fontsize=8, loc="lower left")
plt.tight_layout(); savefig("fig_robustness")

clean_acc = rob[rob["severity"] == 0].set_index("model")["accuracy"]
corr_acc = rob[rob["severity"] > 0].groupby("model")["accuracy"].mean()
rob_summary = pd.DataFrame({"accuracy propre": clean_acc, "accuracy moyenne dégradée": corr_acc})
rob_summary["perte relative"] = 1 - rob_summary["accuracy moyenne dégradée"] / rob_summary["accuracy propre"]
by_kind = rob[rob["severity"] > 0].pivot_table(index="model", columns="corruption", values="accuracy", aggfunc="mean")
rob_summary = rob_summary.join(by_kind).rename(index={**MODEL_LABEL, "avec_aug": "Ablation : avec augmentation", "sans_aug": "Ablation : sans augmentation"})
display(rob_summary.style.format("{:.1%}"))

## 6. Export pour le mobile et la démo
### 6.1 Conversion TensorFlow Lite et mesure sur CPU
La latence GPU ne dit rien d'un téléphone. On convertit chaque modèle en **TFLite** et on mesure, sur **un seul
cœur CPU**, le temps d'inférence d'une image : c'est l'indicateur le plus proche de l'usage embarqué (les valeurs
absolues dépendent de la machine ; ce sont les **rapports entre modèles** qui comptent). Le meilleur modèle est
aussi exporté en version **quantifiée** (poids sur 8 bits) : ≈ 4× plus léger, avec une perte d'accuracy à mesurer.

In [ ]:
def to_tflite(model, out_path, quantize=False):
    try:                                              # voie recommandée avec Keras 3
        sm = str(WORK / "tmp_savedmodel" / f"{model.name}_{int(quantize)}")
        model.export(sm)
        conv = tf.lite.TFLiteConverter.from_saved_model(sm)
    except Exception:
        conv = tf.lite.TFLiteConverter.from_keras_model(model)
    if quantize:
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
    Path(out_path).write_bytes(conv.convert())
    return Path(out_path).stat().st_size / 1e6

def tflite_runner(path, threads=1):
    try:
        from ai_edge_litert.interpreter import Interpreter
    except ImportError:
        Interpreter = tf.lite.Interpreter
    it = Interpreter(model_path=str(path), num_threads=threads); it.allocate_tensors()
    i, o = it.get_input_details()[0]["index"], it.get_output_details()[0]["index"]
    def run(img):
        it.set_tensor(i, img[None].astype(np.float32)); it.invoke()
        return it.get_tensor(o)[0]
    return run

def tflite_eval(path, x, y):
    run = tflite_runner(path); run(x[0])              # échauffement
    t0 = time.perf_counter(); preds = np.array([run(im).argmax() for im in x])
    return float((preds == y).mean()), (time.perf_counter() - t0) / len(x) * 1000

In [ ]:
n_lite = min(300, len(rob_df)); lite_x, lite_y = clean_x[:n_lite], rob_y[:n_lite]
tfl = {}
for name in TRAINED:
    try:
        path = EXPORT_DIR / f"{name}.tflite"
        size = to_tflite(models[name], path)
        acc, ms = tflite_eval(path, lite_x, lite_y)
        tfl[name] = {"tflite_MB": size, "cpu_ms": ms, "tflite_acc": acc}
    except Exception as e:
        print(f"⚠️ Conversion TFLite impossible pour {name} : {type(e).__name__}: {e}")
try:
    qpath = EXPORT_DIR / "model_quant.tflite"
    q_size = to_tflite(models[BEST], qpath, quantize=True)
    q_acc, q_ms = tflite_eval(qpath, lite_x, lite_y)
    QUANT = {"tflite_MB": q_size, "cpu_ms": q_ms, "tflite_acc": q_acc}
except Exception as e:
    QUANT = None; print("⚠️ Quantification impossible :", e)
if BEST in tfl:
    shutil.copy(EXPORT_DIR / f"{BEST}.tflite", EXPORT_DIR / "model.tflite")
shutil.copy(MODEL_DIR / f"{BEST}.keras", EXPORT_DIR / "model.keras")      # secours si TFLite indisponible

tfl_df = pd.DataFrame(tfl).T
results = results.drop(columns=[c for c in tfl_df.columns if c in results.columns]).join(tfl_df)
display(tfl_df.rename(index=MODEL_LABEL).round(3))
if QUANT:
    print(f"Quantifié ({MODEL_LABEL[BEST]}) : {QUANT['tflite_MB']:.1f} Mo · {QUANT['cpu_ms']:.1f} ms · accuracy {QUANT['tflite_acc']:.1%} "
          f"(contre {tfl.get(BEST, {}).get('tflite_acc', float('nan')):.1%} en flottant, sur {n_lite} images)")

In [ ]:
if "cpu_ms" in results and results["cpu_ms"].notna().any():
    d = results.dropna(subset=["cpu_ms"])
    fig, ax = plt.subplots(figsize=(7, 4.2))
    for name, r in d.iterrows():
        ax.scatter(r["cpu_ms"], r["accuracy"], s=60 + 12 * r["tflite_MB"], color=MODEL_COLOR[name],
                   edgecolors=SURFACE, linewidths=1.5, zorder=3)
        ax.annotate(f"{MODEL_LABEL[name]}\n{r['tflite_MB']:.0f} Mo", (r["cpu_ms"], r["accuracy"]),
                    textcoords="offset points", xytext=(10, -4), fontsize=9, color=INK)
    ax.set_xscale("log"); ax.set_xlabel("Latence TFLite sur 1 cœur CPU (ms / image, échelle log)")
    ax.set_ylabel("Accuracy (test)"); ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1, decimals=0))
    ax.margins(x=0.25, y=0.2)
    ax.set_title("Compromis précision / latence (aire du point ∝ taille)")
    savefig("fig_tradeoff")

### 6.2 Fichiers pour l'application et pour le papier
- `export/` : modèle TFLite, noms de classes, **rareté des cartes** (dérivée de la fréquence de la classe dans le
  jeu d'entraînement : une espèce rare dans les données est une carte rare dans le jeu), température et seuils du jeu.
- `paper_assets/` : `results_macros.tex` et `results_table.tex`, lus automatiquement par `paper.tex`.
  **Aucun chiffre n'est recopié à la main** : le papier affiche exactement ce que ce notebook a mesuré.

In [ ]:
freq = train_df["label"].value_counts().reindex(CLASS_NAMES)
rank = freq.rank(pct=True)                       # 0 = classe la plus rare
tier = pd.cut(rank, [0, 0.10, 0.30, 0.60, 1.0], labels=["Légendaire", "Épique", "Rare", "Commune"], include_lowest=True)
XP = {"Commune": 10, "Rare": 20, "Épique": 40, "Légendaire": 80}

class_meta = {"model": BEST, "img_size": IMG_SIZE, "resize_to": CACHE_SIZE, "temperature": TEMP,
              "t_high": T_HIGH, "t_low": T_LOW, "classes": CLASS_NAMES,
              "rarity": {c: str(tier[c]) for c in CLASS_NAMES}, "xp": XP,
              "train_count": {c: int(freq[c]) for c in CLASS_NAMES}}
json.dump(class_meta, open(EXPORT_DIR / "class_meta.json", "w"), ensure_ascii=False, indent=2)

summary = {"n_files": int(n0), "n_clean": int(len(clean)), "n_removed": int(N_REMOVED), "n_classes": int(N_CLASSES),
           "imbalance_ratio": float(IMBALANCE_RATIO), "n_dup_near": N_DUP_NEAR, "n_conflict": N_CONFLICT, "n_unreadable": n_bad,
           "split": {"train": len(train_df), "val": len(val_df), "test": len(test_df)},
           "knn_baseline": float(globals().get("KNN_BASELINE", float("nan"))), "best_hp": BEST_HP, "best_model": BEST,
           "mcnemar_p": float(MCNEMAR_P), "ece_before": float(ECE_BEFORE), "ece_after": float(ECE_AFTER),
           "game": {k: float(v) for k, v in GAME.items()}, "rho_support_f1": float(RHO_SUPPORT),
           "ablation": ablation_df.to_dict("index"), "quant": QUANT,
           "results": results.astype(float).to_dict("index"),
           "robustness": rob_summary.astype(float).to_dict("index")}
json.dump(summary, open(WORK / "results.json", "w"), ensure_ascii=False, indent=2, default=float)

# --- LaTeX : macros + tableau -------------------------------------------------
pct = lambda v: "--" if v is None or (isinstance(v, float) and np.isnan(v)) else f"{100 * v:.1f}".replace(".", "{,}")
num = lambda v, d=1: "--" if v is None or (isinstance(v, float) and np.isnan(v)) else f"{v:.{d}f}".replace(".", "{,}")
get = lambda name, col: results.loc[name, col] if col in results.columns else float("nan")
rs = rob_summary.rename(index={v: k for k, v in MODEL_LABEL.items()})
macros = {
    "nFiles": f"{n0:,}".replace(",", r"\,"), "nClean": f"{len(clean):,}".replace(",", r"\,"), "nRemoved": str(N_REMOVED),
    "nClasses": str(N_CLASSES), "imbalance": num(IMBALANCE_RATIO), "nDup": str(N_DUP_NEAR), "nConflict": str(N_CONFLICT),
    "nTrain": f"{len(train_df):,}".replace(",", r"\,"), "nVal": f"{len(val_df):,}".replace(",", r"\,"), "nTest": f"{len(test_df):,}".replace(",", r"\,"),
    "knnBaseline": pct(summary["knn_baseline"]), "bestModel": MODEL_LABEL[BEST], "runnerUp": MODEL_LABEL[RUNNER_UP],
    "bestAcc": pct(get(BEST, "accuracy")), "bestCiLow": pct(get(BEST, "ci_low")), "bestCiHigh": pct(get(BEST, "ci_high")),
    "bestF": pct(get(BEST, "f1_macro")), "bestTop": pct(get(BEST, "top3")), "bestAuc": num(get(BEST, "auc_macro"), 3),
    "bestCpu": num(get(BEST, "cpu_ms")), "bestSize": num(get(BEST, "tflite_MB")),
    "scratchAcc": pct(get("cnn_scratch", "accuracy")) if "cnn_scratch" in results.index else "--",
    "mcnemarP": num(MCNEMAR_P, 3), "tempT": num(TEMP, 2), "eceBefore": num(100 * ECE_BEFORE), "eceAfter": num(100 * ECE_AFTER),
    "thrHigh": num(T_HIGH, 2), "thrLow": num(T_LOW, 2), "shareCard": pct(GAME["share_card"]), "accCard": pct(GAME["acc_card"]),
    "shareQuiz": pct(GAME["share_quiz"]), "topQuiz": pct(GAME["top3_quiz"]), "shareRescan": pct(GAME["share_rescan"]),
    "robBest": pct(rs.loc[BEST, "perte relative"]) if BEST in rs.index else "--",
    "ablAugVal": pct(ablation["avec_aug"]["val_acc"]), "ablNoAugVal": pct(ablation["sans_aug"]["val_acc"]),
    "ablAugRob": pct(rob_summary.loc["Ablation : avec augmentation", "accuracy moyenne dégradée"]) if "Ablation : avec augmentation" in rob_summary.index else "--",
    "ablNoAugRob": pct(rob_summary.loc["Ablation : sans augmentation", "accuracy moyenne dégradée"]) if "Ablation : sans augmentation" in rob_summary.index else "--",
    "quantSize": num(QUANT["tflite_MB"]) if QUANT else "--", "quantAcc": pct(QUANT["tflite_acc"]) if QUANT else "--",
    "hpLr": f"{BEST_HP['lr']:.1e}".replace(".", "{,}").replace("e-0", r"\times 10^{-").replace("e-", r"\times 10^{-") + "}",
    "hpDropout": num(BEST_HP["dropout"], 2), "hpDense": str(BEST_HP["dense_units"]), "hpSmooth": num(BEST_HP["label_smoothing"], 2),
}
(PAPER_DIR / "results_macros.tex").write_text(
    "% Généré par le notebook HerbaQuest — ne pas modifier à la main\n" +
    "\n".join(f"\\renewcommand{{\\{k}}}{{{v}}}" for k, v in macros.items()) + "\n", encoding="utf-8")

lines = []
for name in results.index:
    cells = [MODEL_LABEL[name], num(get(name, "params_M")), pct(get(name, "accuracy")), pct(get(name, "f1_macro")),
             pct(get(name, "top3")), pct(rs.loc[name, "accuracy moyenne dégradée"]) if name in rs.index else "--",
             num(get(name, "cpu_ms")), num(get(name, "tflite_MB"))]
    if name == BEST:
        cells = [f"\\textbf{{{c}}}" for c in cells]
    lines.append(" & ".join(cells) + r" \\")
(PAPER_DIR / "results_table.tex").write_text("\n".join(lines) + "\n", encoding="utf-8")
with open(PAPER_DIR / "results_macros.tex", "a", encoding="utf-8") as f:   # lignes du tableau, lues par paper.tex
    f.write("\\renewcommand{\\resultsRows}{%\n" + "\n".join(lines) + "}\n")

for f in ("fig_eda_summary", "fig_class_distribution", "fig_confusion", "fig_gradcam", "fig_robustness", "fig_calibration", "fig_tradeoff"):
    if (FIG_DIR / f"{f}.pdf").exists():
        shutil.copy(FIG_DIR / f"{f}.pdf", PAPER_DIR / f"{f}.pdf")
for folder in ("export", "paper_assets", "figures"):
    shutil.make_archive(str(WORK / f"herbaquest_{folder}"), "zip", WORK / folder)
print("Archives à télécharger (onglet Output) :", [f"herbaquest_{f}.zip" for f in ("export", "paper_assets", "figures")])

### 6.3 Démo express dans le notebook
Version minimale pour vérifier le modèle exporté. L'application complète (collection de cartes, XP, quiz) est dans
`app.py` : téléchargez `herbaquest_export.zip`, décompressez-le dans `app/export/`, puis `python app.py`.

In [ ]:
try:
    import gradio as gr

    def classify(image):
        if image is None:
            return {}, "Chargez une photo de plante."
        im = ImageOps.exif_transpose(image).convert("RGB")
        s = CACHE_SIZE / min(im.size)
        im = im.resize((max(CACHE_SIZE, round(im.width * s)), max(CACHE_SIZE, round(im.height * s))), Image.BILINEAR)
        l, t = (im.width - IMG_SIZE) // 2, (im.height - IMG_SIZE) // 2
        x = np.asarray(im.crop((l, t, l + IMG_SIZE, t + IMG_SIZE)), dtype=np.float32)[None]
        p = softmax(predict_logits(models[BEST], x)[0] / TEMP)
        top = np.argsort(-p)[:3]
        verdict = ("🃏 Carte débloquée" if p[top[0]] >= T_HIGH else
                   "❓ Mini-quiz entre les trois propositions" if p[top[0]] >= T_LOW else "🔄 Rescanne sous un autre angle")
        return {CLASS_NAMES[i]: float(p[i]) for i in top}, verdict

    demo = gr.Interface(classify, gr.Image(type="pil", label="Photo de la plante"),
                        [gr.Label(num_top_classes=3, label="Propositions"), gr.Textbox(label="Décision du jeu")],
                        title="HerbaQuest — démo express")
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "Interactive") == "Interactive":
        demo.launch(share=ON_KAGGLE, quiet=True)
    else:
        print("Exécution en arrière-plan : la démo n'est pas lancée. Ouvrez le notebook en mode interactif pour l'essayer.")
except ImportError:
    print("Gradio n'est pas installé : `pip install gradio`, ou utilisez directement app.py.")

## 7. Synthèse et reproductibilité
La cellule suivante récapitule tous les chiffres à interpréter dans le papier et à l'oral. Rien n'est écrit à
l'avance : **l'interprétation dépend de ce que vous obtenez**. Questions à vous poser devant ces chiffres :

1. Le transfert bat-il nettement le CNN de zéro et le plancher k-NN ? De combien ?
2. L'écart entre les deux meilleurs modèles est-il significatif (McNemar) ? Sinon, lequel choisir pour le mobile ?
3. Quelles paires d'espèces sont confondues, et est-ce botaniquement compréhensible ?
4. Grad-CAM : le modèle regarde-t-il la plante ? Sur quelles erreurs regarde-t-il ailleurs ?
5. Quelle perturbation fait le plus de dégâts ? L'augmentation a-t-elle amélioré la robustesse (ablation) ?
6. Après calibration, quelle part des scans débloque une carte, et avec quelle fiabilité ?

**Limites à déclarer** : images du Web et non de vrais scans AR ; étiquettes non validées par des botanistes ;
hyperparamètres cherchés sur une seule architecture ; une seule graine d'entraînement ; latence mesurée sur un
CPU de serveur et non sur un téléphone.

In [ ]:
print("=" * 78, "\nRÉCAPITULATIF HERBAQUEST\n" + "=" * 78)
print(f"Données   : {n0:,} fichiers → {len(clean):,} images après nettoyage ({N_REMOVED} retirées) · {N_CLASSES} classes")
print(f"            déséquilibre max/min = {IMBALANCE_RATIO:.1f} · doublons = {N_DUP_NEAR} · conflits d'étiquette = {N_CONFLICT}")
print(f"Découpage : {len(train_df)} / {len(val_df)} / {len(test_df)} (train / val / test)")
print(f"Plancher  : k-NN sur descripteurs ImageNet non ré-entraînés = {globals().get('KNN_BASELINE', float('nan')):.2%}")
print(f"Optuna    : {BEST_HP}")
print(f"Ablation  : val. avec augmentation {ablation['avec_aug']['val_acc']:.2%} · sans {ablation['sans_aug']['val_acc']:.2%}")
print("\nRésultats sur le test :")
print(results.rename(index=MODEL_LABEL).round(4).to_string())
print(f"\nMcNemar {MODEL_LABEL[BEST]} vs {MODEL_LABEL[RUNNER_UP]} : p = {MCNEMAR_P:.4f}")
print(f"Calibration : T = {TEMP:.2f} · ECE {ECE_BEFORE:.4f} → {ECE_AFTER:.4f}")
print(f"Jeu : carte {GAME['share_card']:.1%} (justes {GAME['acc_card']:.1%}) · quiz {GAME['share_quiz']:.1%} · rescan {GAME['share_rescan']:.1%}")
print("\nEnvironnement :", {"python": os.sys.version.split()[0], "tensorflow": tf.__version__, "keras": keras.__version__,
                            "numpy": np.__version__, "pandas": pd.__version__, "seed": SEED})